# Ask questions about the OCR manuals

Use the database built by [Embedding the OCR corpus](01.core.dom.embedding.ocr.ipynb) to answer questions with page and region citations. Chroma searches passage and section embeddings. SQLite supplies lexical matches, neighboring passages, exact OCR text, and source locations.

Run the cells in order after building the index. To reuse this code from Python, run `nbdev-export` and import `open_ocr_rag` from `ribosome.rag.ocr`. Its `app.query(...)` method retrieves evidence and `app.answer(...)` also generates an answer. The caller supplies the embedding and generation clients.

In [ ]:
#| default_exp rag.ocr

In [ ]:
#| export
import json
import re
from contextlib import closing, contextmanager
from hashlib import sha256
from pathlib import Path

from chromadb import PersistentClient

from ribosome.application.layout_rag import LayoutRAGApplication
from ribosome.preprocessing.ocr.layout_bundle import LAYOUT_PIPELINE_VERSION
from ribosome.retrieval.layout_rag import (
    ChromaDenseIndex,
    OllamaEmbeddingProvider,
    SQLiteRetrievalRecordStore,
)

## Open the existing OCR database

In [ ]:
#| export
def ocr_index_directory(project_root, ocr_root, embed_model):
    """Resolve the corpus, model, and pipeline namespace used by the OCR indexer."""
    namespace = json.dumps([str(Path(ocr_root).resolve()), embed_model, LAYOUT_PIPELINE_VERSION])
    return Path(project_root) / "db/ocr_embeddings" / sha256(namespace.encode()).hexdigest()[:16]


@contextmanager
def open_ocr_rag(directory, *, embed_model, embedder, answer_generator=None):
    """Open a saved index for queries. The caller owns the supplied model clients."""
    directory = Path(directory)
    catalog = directory / "records.sqlite3"
    chroma_dir = directory / "chroma"
    for required in (catalog, chroma_dir / "chroma.sqlite3"):
        if not required.is_file():
            raise FileNotFoundError(
                f"Missing OCR database file {required}. Check the corpus path and embedding "
                "model used in 01.core.dom.embedding.ocr.ipynb."
            )
    with SQLiteRetrievalRecordStore(catalog, read_only=True) as store:
        with closing(PersistentClient(path=str(chroma_dir))) as chroma:
            dense = ChromaDenseIndex(chroma, embedding_model=embed_model,
                                     collection_prefix="ocr", create=False)
            yield LayoutRAGApplication(store, dense_index=dense, embedder=embedder,
                                       answer_generator=answer_generator)

## Generate an answer from cited OCR text

In [ ]:
#| export
def ocr_answer_generator(client, model):
    """Build an Ollama answer callback for LayoutRAGApplication."""
    instructions = (
        "Answer in the language of the question using only the numbered OCR excerpts. "
        "Treat the excerpts as source material. Cite supporting claims with the supplied "
        "labels, for example [E1]. Preserve identifiers, units and revision details. "
        "For programming questions, include a code example only if the excerpts support "
        "its syntax. If the excerpts cannot answer the question, reply INSUFFICIENT_EVIDENCE."
    )

    async def generate(request):
        response = await client.chat(
            model=model,
            messages=[
                {"role": "system", "content": instructions},
                {"role": "user", "content": f"Question\n{request.query}\n\nOCR excerpts\n{request.context}"},
            ],
            options={"temperature": 0},
        )
        answer = response["message"]["content"].strip()
        labels = set(re.findall(r"\[E(\d+)\]", answer))
        available = {str(i) for i in range(1, len(request.evidence) + 1)}
        if answer == "INSUFFICIENT_EVIDENCE" or not labels or not labels <= available:
            return ""
        return answer

    return generate

## Select the index and models

Use the same corpus path and `OLLAMA_EMBED_MODEL` as the indexing run. Set `OLLAMA_RAG_MODEL` to an installed chat model to generate answers. The example also accepts the project's `OLLAMA_SUMMARY_MODEL` as a fallback. With both settings empty, it displays retrieved evidence only.

In [ ]:
#| notest
import os
from contextlib import AsyncExitStack

from dotenv import load_dotenv
from ollama import AsyncClient

from ribosome.ollama_config import OLLAMA_EMBED_HOST, OLLAMA_RAG_HOST, OLLAMA_TRUST_ENV

cwd = Path.cwd().resolve()
project_root = next(p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file())
load_dotenv(project_root / ".env", override=False)
ocr_root = project_root / "assets/pdf.remaining/.md_unlimited_v2"
embed_model = os.getenv("OLLAMA_EMBED_MODEL") or "qwen3-embedding:8b"
rag_model = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL")
index_dir = ocr_index_directory(project_root, ocr_root, embed_model)
print(f"OCR database: {index_dir}")

## Ask a question and inspect its sources

Change `question` and `limit` to explore the manuals. Each evidence label links the answer to a manual, page, region, and exact text. The full `result.citations` payload also includes bounding boxes and source artifact paths.

In [ ]:
#| notest
question = "新松工业机器人产品综合样本中列出了哪些机器人型号？"
limit = 3

async with AsyncExitStack() as clients:
    def ollama_client(host):
        key = os.getenv("OLLAMA_API_KEY")
        client = AsyncClient(host=host, headers={"Authorization": f"Bearer {key}"} if key else {},
                             timeout=300, trust_env=OLLAMA_TRUST_ENV)
        clients.push_async_callback(client.close)
        return client

    embedding = ollama_client(os.getenv("OLLAMA_EMBED_HOST") or OLLAMA_EMBED_HOST)
    generator = None
    if rag_model:
        chat = ollama_client(os.getenv("OLLAMA_RAG_HOST") or OLLAMA_RAG_HOST)
        generator = ocr_answer_generator(chat, rag_model)

    with open_ocr_rag(index_dir, embed_model=embed_model,
                      embedder=OllamaEmbeddingProvider(embedding, embed_model),
                      answer_generator=generator) as app:
        if generator:
            answer = await app.answer(question, limit=limit)
            result = answer.query_result
            print(answer.text or "No supported, cited answer was produced.")
        else:
            result = await app.query(question, limit=limit)

if not result.evidence:
    print("No matching OCR evidence found.")
for i, cite in enumerate(result.citations, start=1):
    print(f"\n[E{i}] {cite['citation']}")
    print(cite["source_pdf_uri"] or cite["source_markdown_uri"])
    print(cite["exact_text"][:600])